In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"

from util.db_helpers import get_mysql_engine
from psrcelmerpy import ElmerGeoConn


#---------------------------------------------
# Config
#---------------------------------------------

# Set to True to force re-downloading tables from MySQL even if a cached
# parquet file already exists in output/pipeline.
FORCE_MYSQL_REFRESH = False
MYSQL_DB = '2023_parcel_baseyear'

In [ ]:
# download tables from MySQL and cache them as parquet files in output/pipeline
output_dir = Path('output/pipeline')
output_dir.mkdir(parents=True, exist_ok=True)

table_names = ['parcels', 'buildings', 'growth_centers','households','jobs','building_types','land_use_types']
dataframes = {}
engine = None

for table in table_names:
    parquet_path = output_dir / f'{table}.parquet'
    if parquet_path.exists() and not FORCE_MYSQL_REFRESH:
        dataframes[table] = pd.read_parquet(parquet_path)
    else:
        if engine is None:
            engine = get_mysql_engine(MYSQL_DB)
        df = pd.read_sql_table(table, engine)
        df.to_parquet(parquet_path)
        dataframes[table] = df

parcels_df = dataframes['parcels']
buildings_df = dataframes['buildings']
growth_centers_df = dataframes['growth_centers']
growth_centers_df['name_lower'] = growth_centers_df['name'].str.lower().str.replace(r'\s+', '', regex=True)
households_df = dataframes['households']
jobs_df = dataframes['jobs']
building_types_df = dataframes['building_types']
land_use_types_df = dataframes['land_use_types']

In [ ]:
eg_conn = ElmerGeoConn()
gdf = eg_conn.read_geolayer('urban_centers')
gdf['name_lower'] = gdf['name'].str.lower().str.replace(r'\s+', '', regex=True)
gdf = gdf[['name_lower','category','acres']]

gdf_ind = eg_conn.read_geolayer('micen')
gdf_ind['name_lower'] = gdf_ind['mic'].str.lower().str.replace(r'\s+', '', regex=True)
gdf_ind['category'] = 'Industrial'
gdf_ind = gdf_ind[['name_lower','category','acres']]

centers_acres = pd.concat([gdf, gdf_ind], ignore_index=True)
growth_centers_df = growth_centers_df.merge(centers_acres, how='left', on='name_lower')
growth_centers_df['acres'] = growth_centers_df['acres'].astype(float)
growth_centers_df = growth_centers_df.rename(columns={'acres':'centers_acres'})

In [ ]:
land_use_include = [
    # 1,	#agriculture
    2,	#Civic and Quasi-Public
    3,	#Commercial
    # 4,	#Fisheries
    # 5,	#Forest, harvestable
    # 6,	#Forest, protected
    7,	#Government
    8,	#Group Quarters
    9,	#Hospital, Convalescent Center
    10,	#Industrial
    11,	#Military
    # 12,	#Mining
    13,	#Mobile Home Park
    14,	#Multi-Family Residential
    15,	#Condo Residential
    16,	#n/a
    17,	#No Land Use Code
    18,	#Office
    # 19,	#Park and Open Space
    20,	#Parking
    # 21,	#Recreation
    # 22,	#Right-of-Way
    23,	#School
    24,	#Single Family Residential
    # 25,	#Transportation, Communication, Utilities
    26,	#Vacant Developable
    # 27,	#Vacant Undevelopable
    28,	#Warehousing
    # 29,	#Water
    30,	#Mixed Use
]

In [ ]:
land_use_res = [
    8,	#Group Quarters
    13,	#Mobile Home Park
    14,	#Multi-Family Residential
    15,	#Condo Residential
    24,	#Single Family Residential
]

In [ ]:
parcels_df['is_residential'] = parcels_df['land_use_type_id'].isin(land_use_res).astype(int)
parcel_mask = (parcels_df['land_use_type_id'].isin(land_use_include)) & (parcels_df['growth_center_id'] > 0)
parcel_sqft = parcels_df.loc[parcel_mask].groupby(['county_id','growth_center_id','is_residential'])['gross_sqft'].sum().reset_index()
parcel_sqft['is_residential'] = parcel_sqft['is_residential'].map({1: 'res', 0: 'non_res'})
parcel_sqft = parcel_sqft.pivot_table(index=['county_id','growth_center_id'], columns='is_residential', values=['gross_sqft'], aggfunc='sum')
parcel_sqft.columns = [f'{col}_{suffix}' for col, suffix in parcel_sqft.columns]
parcel_sqft = parcel_sqft.rename(columns={'gross_sqft_res': 'parcel_gross_sqft_res', 'gross_sqft_non_res': 'parcel_gross_sqft_non_res'})
parcel_sqft = parcel_sqft.reset_index()
parcel_sqft['parcel_gross_sqft_all'] = parcel_sqft['parcel_gross_sqft_res'] + parcel_sqft['parcel_gross_sqft_non_res']
for col in ['parcel_gross_sqft_res', 'parcel_gross_sqft_non_res', 'parcel_gross_sqft_all']:
    parcel_sqft[col] = parcel_sqft[col].round(0).fillna(0).astype(int)

In [ ]:
# add is_residential and growth_center_id columns to buildings_df
is_residential = building_types_df[['building_type_id','is_residential']].set_index('building_type_id')['is_residential']
buildings_df['is_residential'] = buildings_df['building_type_id'].map(is_residential)
parcel_center_xwalk = parcels_df[['parcel_id','growth_center_id']].set_index('parcel_id')['growth_center_id']
buildings_df['growth_center_id'] = buildings_df['parcel_id'].map(parcel_center_xwalk)
county_xwalk = parcels_df[['parcel_id','county_id']].set_index('parcel_id')['county_id']
buildings_df['county_id'] = buildings_df['parcel_id'].map(county_xwalk)

In [ ]:
far = buildings_df.loc[buildings_df['growth_center_id'] != 0].groupby(['county_id','growth_center_id','is_residential'])[['gross_sqft']].sum().reset_index().rename(columns={'gross_sqft': 'bldg_gross_sqft'})
far['is_residential'] = far['is_residential'].map({1: 'res', 0: 'non_res'})
far = far.pivot_table(index=['county_id','growth_center_id'], columns='is_residential', values=['bldg_gross_sqft'], aggfunc='sum')
far.columns = [f'{col}_{suffix}' for col, suffix in far.columns]
far = far.reset_index()
far = far.merge(parcel_sqft, on=['county_id','growth_center_id'], how='left')

In [ ]:
far['bldg_gross_sqft_all'] = far['bldg_gross_sqft_res'] + far['bldg_gross_sqft_non_res']
far = far.merge(growth_centers_df[['growth_center_id','centers_acres']], on='growth_center_id', how='left')
far['centers_sqft'] = (far['centers_acres'] * 43560).round(0).fillna(0).astype(int)
for type in ['res','non_res','all']:
    far[f'far_{type}'] = far[f'bldg_gross_sqft_{type}'] / far[f'parcel_gross_sqft_{type}']

In [ ]:
building_parcel_xwalk = buildings_df[['building_id','parcel_id']].set_index('building_id')['parcel_id']
households_df['parcel_id'] = households_df['building_id'].map(building_parcel_xwalk)
households_df['growth_center_id'] = households_df['parcel_id'].map(parcel_center_xwalk)
households_df['county_id'] = households_df['parcel_id'].map(county_xwalk)
persons = households_df.loc[households_df['growth_center_id'] != 0].groupby(['county_id','growth_center_id'])[['persons']].sum().reset_index()

In [ ]:
jobs_df['parcel_id'] = jobs_df['building_id'].map(building_parcel_xwalk)
jobs_df['growth_center_id'] = jobs_df['parcel_id'].map(parcel_center_xwalk)
jobs_df['county_id'] = jobs_df['parcel_id'].map(county_xwalk)
jobs_mask = (jobs_df['growth_center_id'] != 0) & (jobs_df['home_based_status'] == 0)
jobs = jobs_df.loc[jobs_mask].groupby(['county_id','growth_center_id']).size().reset_index(name='jobs')

In [ ]:
au = persons.merge(jobs, on=['county_id','growth_center_id'], how='outer').merge(far, on=['county_id','growth_center_id'], how='outer')
au['activity_units'] = au['persons'] + au['jobs']
au['au_acre'] = au['activity_units'] / au['centers_acres']
au['pop_acre'] = au['persons'] / au['centers_acres']
au['jobs_acre'] = au['jobs'] / au['centers_acres']

In [ ]:
county_map = {
    33: 'King',
    35: 'Kitsap',
    53: 'Pierce',
    61: 'Snohomish'
}
au['county'] = au['county_id'].map(county_map)

au = au.merge(growth_centers_df[['growth_center_id','name','category']], on='growth_center_id', how='left')
au['is_industrial'] = (au['category'] == 'Industrial').astype(int)
au['parcel_sqft_pct'] = ((au['parcel_gross_sqft_all'] / au['centers_sqft']) * 100).round(1)

# Residential plot

In [ ]:
#| title: "Population per Acre vs. Residential FAR"


valid = au.loc[(au.is_industrial == 0) & (au.persons > 1000)][['county', 'name', 'far_res', 'pop_acre']].dropna()

slope, intercept = np.polyfit(valid['far_res'], valid['pop_acre'], 1)
x_line = np.linspace(valid['far_res'].min(), valid['far_res'].max(), 100)

fig = px.scatter(
    valid,
    x='far_res',
    y='pop_acre',
    hover_name='name',
    hover_data={'county': True, 'far_res': ':.2f', 'pop_acre': ':.1f'},
    labels={'far_res': 'FAR', 'pop_acre': 'Population / Acre'},
)
# Assign (rather than call as a bare statement) so these chainable methods'
# return values aren't treated as separate top-level expression outputs.
# Quarto dashboards set Jupyter's shell interactivity to "all" so every
# top-level expression is displayed - since Plotly's add_trace/update_layout
# return `self`, calling them as bare statements was producing extra charts.
fig = fig.add_trace(go.Scatter(
    x=x_line,
    y=slope * x_line + intercept,
    mode='lines',
    name=f'y = {slope:.2f}x + {intercept:.2f}',
    line=dict(color='red'),
))
fig = fig.update_layout(
    xaxis_title='Residential FAR',
    yaxis_title='Population / Acre',
    legend_title_text='',
)
fig

# Non-Residential plot

In [ ]:
#| title: "Jobs per Acre vs. Non-Residential FAR"
valid = au[['county', 'name', 'far_non_res', 'jobs_acre']].dropna()

slope, intercept = np.polyfit(valid['far_non_res'], valid['jobs_acre'], 1)
x_line = np.linspace(valid['far_non_res'].min(), valid['far_non_res'].max(), 100)

fig = px.scatter(
    valid,
    x='far_non_res',
    y='jobs_acre',
    hover_name='name',
    hover_data={'county': True, 'far_non_res': ':.2f', 'jobs_acre': ':.1f'},
    labels={'far_non_res': 'FAR', 'jobs_acre': 'Jobs / Acre'},
)
# Assign (rather than call as a bare statement) so these chainable methods'
# return values aren't treated as separate top-level expression outputs.
# Quarto dashboards set Jupyter's shell interactivity to "all" so every
# top-level expression is displayed - since Plotly's add_trace/update_layout
# return `self`, calling them as bare statements was producing extra charts.
fig = fig.add_trace(go.Scatter(
    x=x_line,
    y=slope * x_line + intercept,
    mode='lines',
    name=f'y = {slope:.2f}x + {intercept:.2f}',
    line=dict(color='red'),
))
fig = fig.update_layout(
    xaxis_title='Non-Residential FAR',
    yaxis_title='Jobs / Acre',
    legend_title_text='',
)
fig

# Non-Res & Res plot

In [ ]:
#| title: "Activity Units per Acre vs. FAR"
valid = au[['county', 'name', 'far_all', 'au_acre']].dropna()

slope, intercept = np.polyfit(valid['far_all'], valid['au_acre'], 1)
x_line = np.linspace(valid['far_all'].min(), valid['far_all'].max(), 100)

fig = px.scatter(
    valid,
    x='far_all',
    y='au_acre',
    hover_name='name',
    hover_data={'county': True, 'far_all': ':.2f', 'au_acre': ':.1f'},
    labels={'far_all': 'FAR', 'au_acre': 'AU / Acre'},
)
# Assign (rather than call as a bare statement) so these chainable methods'
# return values aren't treated as separate top-level expression outputs.
# Quarto dashboards set Jupyter's shell interactivity to "all" so every
# top-level expression is displayed - since Plotly's add_trace/update_layout
# return `self`, calling them as bare statements was producing extra charts.
fig = fig.add_trace(go.Scatter(
    x=x_line,
    y=slope * x_line + intercept,
    mode='lines',
    name=f'y = {slope:.2f}x + {intercept:.2f}',
    line=dict(color='red'),
))
fig = fig.update_layout(
    xaxis_title='FAR',
    yaxis_title='AU / Acre',
    legend_title_text='',
)
fig

# FAR Regression

In [ ]:
far_features = ['pop_acre', 'jobs_acre','is_industrial']
far_reg_df = au[far_features + ['far_all']].replace([np.inf, -np.inf], np.nan).dropna()

X = sm.add_constant(far_reg_df[far_features])
y = far_reg_df['far_all']
far_model = sm.OLS(y, X).fit()
far_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                far_all   R-squared:                       0.973
Model:                            OLS   Adj. R-squared:                  0.971
Method:                 Least Squares   F-statistic:                     425.2
Date:                Mon, 03 Aug 2026   Prob (F-statistic):           1.39e-27
Time:                        11:14:30   Log-Likelihood:                 4.0054
No. Observations:                  39   AIC:                          -0.01079
Df Residuals:                      35   BIC:                             6.643
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -0.2069      0.062     -3.331      0.002      -0.333      -0.081
pop_acre          0.0351      0.004      8.624      0.000       0.027       0.043
jobs_acre         0.0210      0.001     17.360      0.000       0.019       0.023
is_industrial     0.3268      0.096      3.420      0.002       0.133       0.521
==============================================================================
Omnibus:                        1.892   Durbin-Watson:                   1.487
Prob(Omnibus):                  0.388   Jarque-Bera (JB):                0.927
Skew:                          -0.275   Prob(JB):                        0.629
Kurtosis:                       3.519   Cond. No.                         164.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

# FAR Table

In [ ]:
#| title: "FAR by Growth Center"
au[['county','name','bldg_gross_sqft_non_res', 'bldg_gross_sqft_res', 'bldg_gross_sqft_all',
    'parcel_gross_sqft_res', 'parcel_gross_sqft_non_res', 'parcel_gross_sqft_all',
    'far_res', 'far_non_res', 'far_all']]

,county,name,bldg_gross_sqft_non_res,bldg_gross_sqft_res,bldg_gross_sqft_all,parcel_gross_sqft_res,parcel_gross_sqft_non_res,parcel_gross_sqft_all,far_res,far_non_res,far_all
0,King,Auburn,3326850.0,1768626.0,5095476.0,3659903,6912908,10572811,0.483244,0.481252,0.481941
1,King,Bellevue,40617329.0,10636173.0,51253502.0,1090863,12059491,13150355,9.750237,3.368080,3.897499
2,King,Burien,3381113.0,1806127.0,5187240.0,3559497,9311390,12870888,0.507411,0.363116,0.403021
3,King,Federal Way,2494599.0,261635.0,2756234.0,106449,8148415,8254864,2.457844,0.306145,0.333892
4,King,Kent,2994430.0,1186542.0,4180972.0,855499,7123485,7978984,1.386959,0.420360,0.523998
5,King,Kirkland Totem Lake,7743882.0,5871758.0,13615640.0,9551974,18237340,27789314,0.614717,0.424617,0.489960
6,King,Redmond Downtown,5805982.0,8424325.0,14230307.0,2019323,14400407,16419730,4.171856,0.403182,0.866659
7,King,Redmond-Overlake,24173230.0,5922963.0,30096193.0,3318892,27222967,30541859,1.784621,0.887972,0.985408
8,King,Renton,10509394.0,3814252.0,14323646.0,2638385,17029149,19667534,1.445677,0.617141,0.728289
9,King,SeaTac,9012023.0,4138448.0,13150471.0,11253354,16442802,27696156,0.367752,0.548083,0.474812


# Activity Units Table

In [ ]:
#| title: "Activity Units by Growth Center"
au[['county','name','jobs','persons','activity_units','centers_acres','au_acre','pop_acre','jobs_acre']]

,county,name,jobs,persons,activity_units,centers_acres,au_acre,pop_acre,jobs_acre
0,King,Auburn,5354,3612.0,8966.0,354.798381,25.270690,10.180430,15.090260
1,King,Bellevue,52050,16544.0,68594.0,410.182058,167.228182,40.333310,126.894872
2,King,Burien,3758,3851.0,7609.0,418.958276,18.161713,9.191846,8.969867
3,King,Federal Way,2737,604.0,3341.0,219.073287,15.250604,2.757068,12.493536
4,King,Kent,6091,2153.0,8244.0,292.357588,28.198345,7.364269,20.834075
5,King,Kirkland Totem Lake,15575,8227.0,23802.0,841.904117,28.271628,9.771897,18.499731
6,King,Redmond Downtown,11142,9678.0,20820.0,508.761699,40.922892,19.022658,21.900234
7,King,Redmond-Overlake,60566,7002.0,67568.0,863.370556,78.260718,8.110075,70.150643
8,King,Renton,16601,5480.0,22081.0,605.662871,36.457576,9.047938,27.409638
9,King,SeaTac,26970,11483.0,38453.0,884.700414,43.464431,12.979535,30.484896


# Sqft Table

In [ ]:
au[['county','name','category','centers_sqft','parcel_gross_sqft_all','parcel_sqft_pct']]

,county,name,category,centers_sqft,parcel_gross_sqft_all,parcel_sqft_pct
0,King,Auburn,Urban,15455017,10572811,68.4
1,King,Bellevue,Metro,17867530,13150355,73.6
2,King,Burien,Urban,18249822,12870888,70.5
3,King,Federal Way,Urban,9542832,8254864,86.5
4,King,Kent,Urban,12735097,7978984,62.7
5,King,Kirkland Totem Lake,Urban,36673343,27789314,75.8
6,King,Redmond Downtown,Urban,22161660,16419730,74.1
7,King,Redmond-Overlake,Metro,37608421,30541859,81.2
8,King,Renton,Urban,26382675,19667534,74.5
9,King,SeaTac,Urban,38537550,27696156,71.9


# Notes

- data source: urbansim 2023_parcel_baseyear data
- Activity units = persons (from households table) + jobs
- home-based jobs are not included
- Activity units per acre = activity units / centers_acres / 43560
- Centers_acres uses entire growth center polygons (includes water/parks/open space/undevelopable land/ROW)
- FAR is calculated as bldg_gross_sqft / parcel_gross_sqft
- parcel_gross_sqft does not include water/parks/open space/undevelopable land/ROW